# 2026-03-18 (수) - Embedding Cache & Vector Store 기초

**W2 Day 2: 캐시, 배치 처리, 시각화, FAISS**

어제(3/17) 이어서 - Embedding을 실전에서 써먹는 법.

오늘 배울 내용:
- **CacheBackedEmbeddings**로 비용/시간 절감 실측
- **배치 처리** 함수로 대량 문서 임베딩
- **코사인 유사도 실전 예제** (반의어, 다국어 등 엣지 케이스)
- **차원 축소 시각화**: PCA vs t-SNE로 1536→2차원
- **FAISS**: Facebook AI Similarity Search - 벡터 스토어 입문
- **Document + metadata**: 문서에 카테고리/출처 등 부가정보 실기
- **다국어(multilingual) 임베딩**: 한↔영 교차 검색

핵심 비유:
- 캐시 = 자주 보는 책을 집 책장에 꽂아두기
- 배치 = 여러 장을 한 번에 복사기에 넣기 (API 호출 줄이기)
- PCA = 공간을 "가장 중요한 두 축"으로 납작하게 누르기
- t-SNE = 가까운 점은 가깝게, 먼 점은 멀게 재배치
- FAISS = 수백만 벡터 검색용 초고속 도서관 사서

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w2_token_embedding_vector_rag/llm_260318_embedding_cache.ipynb)

## 0. Colab 환경 설정

Colab에서 실행 시 필요한 패키지들.

In [ ]:
# Colab 환경: 필요한 패키지 설치
!pip install -q langchain-text-splitters langchain-openai langchain_classic langchain_community faiss-cpu scikit-learn seaborn

In [ ]:
# 기본 라이브러리 임포트
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv

# LangChain LLM / Embedding / Splitter 임포트
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Colab이면 userdata, 로컬이면 .env
try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except ImportError:
    load_dotenv()
    api_key = os.getenv('OPENAI_API_KEY')

In [ ]:
# LLM & Embedding 모델 초기화
# gpt-4o-mini: 저렴한 chat 모델
# text-embedding-3-small: 1536차원 임베딩, 다국어 지원
llm = ChatOpenAI(model='gpt-4o-mini', api_key=api_key)
embeddings_model = OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)

## 1. Embedding Cache - 실측

어제 마지막에 setup만 했던 CacheBackedEmbeddings, 오늘은 **실제로 얼마나 빨라지나** 측정.

### 작동 방식
1. `embed_documents(["텍스트A", "텍스트B"])` 호출
2. 내부적으로 텍스트의 SHA-1 해시를 키로 ByteStore에서 조회
3. **캐시 HIT**: ByteStore에서 즉시 리턴 (API 호출 X, 비용 0)
4. **캐시 MISS**: OpenAI API 호출 → 결과를 ByteStore에 저장 후 리턴

> **비유**: 스타벅스 앱에서 자주 시키는 음료를 '즐겨찾기'로 저장해두면 다음엔 한 번 탭으로 주문. 메뉴판 훑을 필요 X.

In [ ]:
# 테스트 임베딩 한 번 찍어 API 키 동작 확인
text_emb = embeddings_model.embed_query('Hello')

In [ ]:
# 캐시용 클래스 임포트
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_core.stores import InMemoryByteStore

In [ ]:
# 1) InMemoryByteStore: 프로세스 메모리(RAM)에 캐시 저장
#    - 프로세스 종료 시 사라짐 (영구 저장 아님)
#    - 영구 저장 원하면 SQLite/File 등 다른 ByteStore 교체
store = InMemoryByteStore()

# 2) CacheBackedEmbeddings: 기존 embedding 모델을 감싸서 캐시 기능 부여
#    namespace: 여러 모델을 하나의 store에 저장할 때 충돌 방지
cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model, store, namespace="embedding-cache"
)

In [ ]:
# 기본 동작 확인: 2개 문서 -> 2개 1536차원 벡터
texts = ['인공지능이 계속 발전하고 있습니다.', 'Langchain으로 RAG를 구현합시다.']
vecs1 = cached_embeddings.embed_documents(texts)
len(vecs1[0]), len(vecs1[1])

In [ ]:
# 캐시 효과 측정: 같은 텍스트 두 번 임베딩
# 1차: API 호출 발생 -> 수십~수백 ms
# 2차: 캐시에서 즉시 리턴 -> 1ms 수준
start = time.time()
texts = ["오늘 점심에는 라면을 먹었습니다", "내일 저녁에는 햄버거를 먹을겁니다"]
vecs1 = cached_embeddings.embed_documents(texts)
t1 = time.time() - start
print(f"1차 call, {t1}초")

start = time.time()
vecs2 = cached_embeddings.embed_documents(texts)
t2 = time.time() - start
print(f"2차 call, {t2}초")

### 실측 결과 해석

- 1차 호출: OpenAI API 왕복 시간 + 네트워크 지연
- 2차 호출: 해시 조회 + 바이트 디코딩만 → **수백 배 빠름**
- 매번 정확히 같은 값이 나오는 이유는 임베딩이 **결정론적(deterministic)** 이기 때문

캐시 저장 위치: **RAM (메모리)** → 프로세스 종료 시 휘발.
영구 저장하려면 SQLite 등 ByteStore를 교체해서 쓰면 됨.

## 2. 배치 처리 함수 - `process_documents`

### 왜 배치?

문서 100개를 하나씩 돌리면 API를 100번 호출 → 느리고 비쌈.
**배치(batch_size=5)** 로 묶어 한 번에 5개씩 넘기면 API 호출 20번으로 끝.

> **비유**: 복사기에 A4 한 장씩 20번 돌리는 대신, 5장씩 4번 묶어서 돌리기.

### 구현 포인트
- `range(0, len(docs), batch_size)`: 0, 5, 10, 15... 처럼 batch_size만큼 건너뛰며 순회
- 배치마다 토큰 수 집계 → 비용 추정 가능
- `list.extend()`: 결과 벡터 리스트를 이어붙임 (append와 다름!)
  - `append([1,2])` → `[[1,2]]`
  - `extend([1,2])` → `[1, 2]`

In [ ]:
# [내 풀이] 배치마다 실행시간 + 처리 토큰 출력, 전체 완료 시 총합 출력
import time
import tiktoken

def process_documents(docs, embedding_model, batch_size=5):
    all_vectors = []
    total_tokens = 0
    total_start = time.time()

    # text-embedding-3-small 모델용 tokenizer 로드
    enc = tiktoken.encoding_for_model("text-embedding-3-small")

    # 0, batch_size, 2*batch_size, ... 로 건너뛰며 배치 생성
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        batch_start = time.time()

        # 배치 단위로 임베딩 API 호출
        vectors = embedding_model.embed_documents(batch)
        all_vectors.extend(vectors)  # 리스트 확장 (append 아님!)

        # 토큰 개수 집계
        batch_tokens = sum(len(enc.encode(doc)) for doc in batch)
        total_tokens += batch_tokens

        batch_time = time.time() - batch_start
        print(f"배치 {i // batch_size + 1} 완료 | 소요시간: {batch_time:.4f}초 | 처리 토큰: {batch_tokens}")

    total_time = time.time() - total_start
    print(f"\n전체 완료 | 총 소요시간: {total_time:.4f}초 | 전체 처리 토큰: {total_tokens}")

    return all_vectors

In [ ]:
# AI 관련 짧은 문서 10개로 배치 처리 테스트
documents = [
    "AI는 대량의 데이터를 학습하여 패턴을 인식한다.",
    "머신러닝은 AI의 한 분야로, 경험을 통해 성능을 향상시킨다.",
    "딥러닝은 신경망을 여러 층으로 쌓아 복잡한 문제를 해결한다.",
    "자연어처리(NLP)는 컴퓨터가 인간의 언어를 이해하도록 돕는다.",
    "AI 교육은 논리적 사고력과 문제 해결 능력을 함께 키워준다.",
    "챗봇은 자연어처리 기술을 활용한 대화형 AI 시스템이다.",
    "강화학습은 보상과 패널티를 통해 AI가 최적의 행동을 학습한다.",
    "컴퓨터 비전은 이미지와 영상을 분석하는 AI 기술이다.",
    "AI 윤리는 인공지능 기술의 공정하고 책임 있는 사용을 다룬다.",
    "프롬프트 엔지니어링은 AI 모델에서 원하는 결과를 얻는 기술이다.",
]

vectors = process_documents(documents, embeddings_model, batch_size=2)

In [ ]:
# 두 번째 호출 - 캐시 효과 확인용 (embeddings_model을 그대로 쓰면 캐시 안됨)
# 만약 cached_embeddings로 바꿔 호출하면 극적으로 빨라짐
vectors = process_documents(documents, embeddings_model, batch_size=2)

In [ ]:
# 배치 크기를 5로 늘렸을 때 - API 호출 수 감소로 조금 더 빠름
vectors = process_documents(documents, embeddings_model, batch_size=5)

In [ ]:
# 동일 호출 반복 - OpenAI API는 서버 쪽 캐시/가속이 걸릴 수도 있음
vectors = process_documents(documents, embeddings_model, batch_size=5)

In [ ]:
# [선생님 답안 버전] - 코드 정리본
import tiktoken

def count_token(text, model='gpt-4o-mini'):
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

In [ ]:
# 선생님 버전의 process_documents: progress 표시 + 깔끔한 출력
def process_documents2(docs, embedding_model, batch_size=5):
    all_vectors = []
    total_tokens = 0
    start = time.time()

    for i in range(0, len(docs), batch_size):
        batch = docs[i:i+batch_size]  # 0~4, 5~9 ...
        batch_tokens = sum(count_token(doc) for doc in batch)
        total_tokens += batch_tokens

        vectors = embedding_model.embed_documents(batch)
        all_vectors.extend(vectors)  # append(X), extend(O)

        # 진행률 표시 - len(docs)가 배치 사이즈로 딱 안 나눠지는 경우 대비 min()
        progress = min(i + batch_size, len(docs))
        print(f' [{progress} / {len(docs)}] batch {i // batch_size + 1} process ({batch_tokens} tokens)')

    elapsed = time.time() - start
    print(f'completed, {len(docs)} documents, {total_tokens}')
    print(f' {elapsed} seconds ')

    return all_vectors

In [ ]:
# 100개짜리 테스트 문서 생성 후 배치 처리
documents2 = [f'doc {i} : 이것은 테스트 문서입니다. 인공지능 관련... ' for i in range(100)]
vectors2 = process_documents2(documents2, embeddings_model, batch_size=5)

## 3. Cosine Similarity 실전 - 엣지 케이스

어제 이론은 배웠고, 오늘은 "어? 왜 이런 결과가?" 싶은 케이스들을 직접 관찰.

### 예상 vs 실제

- "날씨가 좋다" vs "날씨가 나쁘다" → **의미 반대인데 유사도 높음** (같은 주제 = 날씨)
- "파이썬 프로그래밍" vs "python programming" → 뜻 같은데 **낮게 나옴** (언어 다름)
- 임베딩은 **"문맥/주제"** 를 담지 **"긍정/부정 뉘앙스"** 를 완벽히 구분하진 못함

### 정적 임베딩 vs 동적 임베딩
- **정적(TF-IDF, Word2Vec)**: 단어 빈도·위치만 봄 → 같은 단어 많이 겹치면 유사도 높음
- **동적(현대 LLM 임베딩)**: 문맥까지 학습 → 의미적 유사성까지 반영
  하지만 감정/극성(polarity)은 여전히 모호할 수 있음

In [ ]:
# Cosine Similarity 코사인 유사도
# 자연어를 벡터로 임베딩해서 연산을 통한 결과 도출
# 대부분 벡터 스페이스에서 벡터로 존재하며 벡터들의 관계가 어떻게 연관되어 있는지
# Cosine 0도: 1 (가장 유사), 180도: -1 (정반대)
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 문장 쌍 예시 - 기본 케이스
pairs = [
    ('오늘 날씨가 참 좋다.', '오늘은 날씨가 화창한 날이다.'),
    ('오늘 날씨가 참 좋다.', '오늘은 날씨가 안좋다.'),  # 반대 의미인데?
    ('파이썬 프로그래밍', 'python programming'),
    ('인공지능 기술의 발전', '엄마가 끓여주는 김치찌개'),
    ('서울은 한국의 수도이다.', 'Seoul is the capital city in Korea')
]

# 더 다양한 케이스 - 반의어, 신조어, 교차언어 등
pairs2 = [
    ('이 치킨 진짜 맛없다', '이 치킨 진짜 맛있다'),
    ('배고파 죽겠다', '밥 언제 먹어'),
    ('나 너 좋아해', 'I like you'),
    ('완전 핵노잼이다', '정말 재미없다'),
    ('오늘 주식이 폭락했다', '강아지가 공을 물고 뛴다'),
    ('시험 붙었다 너무 기뻐', '시험 떨어졌다 너무 슬퍼'),
    ('그 사람은 여우 같다', '그 사람은 교활하다'),
    ('그 영화 레전드다 ㄹㅇ', '그 영화 정말 훌륭했습니다'),
    ('금리가 올랐다', '대출 이자가 부담된다'),
    ('인간은 본래 선하다', '인간은 본래 악하다'),
]

def pair_similar_decoder(pairs):
    # 각 쌍을 임베딩 -> 두 벡터 사이 코사인 유사도 출력
    for text_a, text_b in pairs:
        embs = embeddings_model.embed_documents([text_a, text_b])
        simi = cosine_similarity([embs[0]], [embs[1]])[0][0]
        print(f'similarity: {simi}, {text_a}, {text_b}')

In [ ]:
# 기본 쌍 + 심화 쌍 비교 -> 모델의 "감정" 구분 한계 관찰
pair_similar_decoder(pairs)
print(' ===== ')
pair_similar_decoder(pairs2)

## 4. 유사 문장 검색 - `find_most_similar`

쿼리를 받아 코퍼스에서 가장 비슷한 문장을 찾기.

### 두 가지 구현
1. **하나씩 임베딩** (비효율): 쿼리 + N개 코퍼스 → API 호출 N+1번
2. **일괄 임베딩** (효율): 쿼리와 코퍼스를 한 리스트로 합쳐 **한 번에 호출**

> **비유**: 지도에서 내 위치와 N개 가게 거리 재기.
> 1번은 "이 가게 거리? 저 가게 거리?" 매번 지도 앱 켜는 것,
> 2번은 내 위치+모든 가게를 한 번에 띄워놓고 가까운 순으로 정렬.

In [ ]:
# vectorize 후, query와의 cosine-similarity를 계산해서
# return 가장 유사한 문장을 리턴하세요.
corpus = [
    "LangChain은 LLM 앱 개발 프레임워크입니다",
    "RAG는 검색 증강 생성 기술입니다",
    "벡터 데이터베이스는 임베딩을 저장합니다",
    "파인튜닝은 모델을 특정 데이터에 맞게 조정합니다",
    "프롬프트 엔지니어링은 AI에게 좋은 지시를 작성하는 기술입니다",
    "LangChain으로 챗봇 만들기",
    "React vs Vue 성능 비교",
    "GPT API 활용 가이드",
]
query = 'AI 챗봇 개발하고 싶어요'

# [내 풀이 v1] 하나씩 임베딩 - 비효율 버전
def find_most_similar(query, corpus):
    query_emb = embeddings_model.embed_documents([query])[0]

    best_score = -1
    best_sentence = None

    # 코퍼스 각 문장을 매번 embed_documents 호출 -> API 많이 씀 (비효율!)
    for c in corpus:
        c_emb = embeddings_model.embed_documents([c])[0]
        simi = cosine_similarity([query_emb], [c_emb])[0][0]
        print(f'similarity: {simi:.4f} | {c}')

        if simi > best_score:
            best_score = simi
            best_sentence = c

    print(f'\nMost Similar Sentense: {best_sentence} ({best_score:.4f})')
    return best_sentence

find_most_similar(query, corpus)

In [ ]:
# 다른 쿼리로 테스트
query2 = 'GEMINI가 좋아요? GPT가 좋아요?'
find_most_similar(query, corpus)

In [ ]:
# [내 풀이 v2] 일괄 임베딩 + 정렬 - 효율 버전
def find_most_similar2(query, corpus):
    # 쿼리와 코퍼스를 합쳐 한 번에 임베딩 (API 호출 1번!)
    all_texts = [query] + corpus
    all_embs = embeddings_model.embed_documents(all_texts)
    query_emb = all_embs[0]
    corpus_emb = all_embs[1:]

    # 쿼리 vs 코퍼스 전체 유사도 계산 (1 x N 행렬 -> [0] 꺼내서 N 벡터)
    sims = cosine_similarity([query_emb], corpus_emb)[0]
    # (인덱스, 유사도) 튜플로 만들고 유사도 내림차순 정렬
    ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)

    for rank, (idx, sim) in enumerate(ranked):
        print(f'{rank} sim : {sim}, {corpus[idx]}')

    return ranked[0]

find_most_similar2(query2, corpus)

## 5. 임베딩 시각화 - PCA vs t-SNE

1536차원은 사람 눈으로 못 봄. 2차원(or 3차원)으로 **압축해서 시각화**하자.

### PCA (Principal Component Analysis, 주성분 분석)
- 데이터가 **가장 많이 퍼진 방향**(분산 최대 축)을 찾아 그 축들로 좌표계를 재구성
- 선형 변환 → **빠름**, 항상 같은 결과 (결정론적)
- 전체적인 분포를 보기 좋음, 가까운 점 묶음은 다소 흐릿

> **비유**: 복잡한 조각상을 가장 넓게 보이는 각도(카메라 각도) 두 개만 골라 투영. 전체 윤곽은 보임.

### t-SNE (t-distributed Stochastic Neighbor Embedding)
- 점들 사이의 **거리 관계**를 유지하며 축소
- 비선형, 확률적(t-분포 사용) → **느림**, 실행할 때마다 조금씩 달라짐 (random_state로 고정 가능)
- **군집이 뚜렷**하게 보임, 시각적으로 예쁨
- `perplexity`: 각 점 주변 몇 개를 이웃으로 볼지 (작으면 로컬, 크면 글로벌)

> **비유**: 반 친구들끼리 친한 정도대로 자리를 다시 배치. 친한 애들은 옆자리, 안 친한 애들은 먼 자리.

In [ ]:
# Embedding 시각화 - 1536차원 -> 2차원 압축
# PCA: 분산 큰 축 기준 선형 압축 (빠름, 재현성 100%)
from sklearn.decomposition import PCA
# t-SNE: 거리 관계 유지 비선형 압축 (느림, 랜덤성 있음 -> random_state 고정 필요)
from sklearn.manifold import TSNE

# 4개 토픽 x 4문장씩 = 16개 샘플 문장
topic_sentences = {
    "tech": [
        "인공지능이 산업을 혁신하고 있다",
        "클라우드 컴퓨팅이 IT를 변화시킨다",
        "블록체인이 금융을 바꾸고 있다",
        "5G가 통신을 혁신한다",
    ],
    "sports": [
        "축구 경기에서 역전골이 터졌다",
        "올림픽에서 금메달을 획득했다",
        "야구 시즌이 개막했다",
        "마라톤 대회에서 신기록을 세웠다",
    ],
    "food": [
        "이 레스토랑의 파스타가 맛있었다",
        "한국 김치는 발효 식품이다",
        "새로운 카페에서 커피를 마셨다",
        "맛집 탐방이 요즘 인기다",
    ],
    "finance": [
        "주식 시장이 크게 하락했다",
        "비트코인 가격이 급등했다",
        "금리 인상이 예상된다",
        "환율이 급변하고 있다",
    ],
}

In [ ]:
# 모든 문장을 하나의 리스트로 평평하게 펴고, 라벨도 같은 순서로 만듦
all_texts = []
all_labels = []

for topic, sents in topic_sentences.items():
    all_texts.extend(sents)  # 문장 리스트를 그대로 이어붙임
    all_labels.extend([topic] * len(sents))  # 각 문장에 토픽 라벨 매핑

all_labels

In [ ]:
# 16개 문장 일괄 임베딩 -> 16 x 1536 행렬
all_embs = embeddings_model.embed_documents(all_texts)
len(all_embs), len(all_embs[0])

In [ ]:
# numpy 배열로 변환 - PCA/t-SNE가 행렬 입력을 요구하므로
# shape: (16, 1536) - 16개 샘플, 1536차원
import numpy as np
emb_matrix = np.array(all_embs)
emb_matrix

In [ ]:
# PCA로 2차원 축소
# n_components=2: 2개의 주성분(principal component)만 찾아서 2차원으로
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(emb_matrix)  # (16, 1536) -> (16, 2)
coords_2d

In [ ]:
# PCA 결과 시각화 - 토픽별 색깔 다르게
import matplotlib.pyplot as plt

colors = {"tech": "#FF6B6B", "sports": "#4ECDC4", "food": "#FFD93D", "finance": "#6C5CE7"}
fig, ax = plt.subplots(figsize=(12, 8))

for topic in topic_sentences:
    # 현재 topic에 해당하는 인덱스만 True로 마스킹
    mask = [l == topic for l in all_labels]
    x = coords_2d[mask, 0]  # 0번째 주성분 (가장 분산 큰 축)
    y = coords_2d[mask, 1]  # 1번째 주성분
    ax.scatter(x, y, c=colors[topic], label=topic, s=100)

ax.legend(fontsize=12)
ax.set_title('PCA 2D - 모델이 바라보는 데이터의 압축된 축소판')
plt.show()

In [ ]:
# t-SNE로 2차원 축소
# random_state=42: 매번 같은 결과가 나오도록 시드 고정
# perplexity=5: 각 점 주변 5개 이웃을 보고 배치 (16개 샘플에 적당)
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
coords_tsne = tsne.fit_transform(emb_matrix)

fig, ax = plt.subplots(figsize=(12, 8))
for topic in topic_sentences:
    mask = [l == topic for l in all_labels]
    x = coords_tsne[mask, 0]
    y = coords_tsne[mask, 1]
    ax.scatter(x, y, c=colors[topic], label=topic, s=100)

ax.legend(fontsize=12)
ax.set_title('t-SNE - 같은 토픽끼리 더 뚜렷하게 뭉침')
plt.show()

### PCA vs t-SNE 정리

| 특성 | PCA | t-SNE |
|---|---|---|
| 방식 | 선형 (분산 축 찾기) | 비선형 (거리 보존) |
| 속도 | 빠름 | 느림 |
| 재현성 | 100% 동일 결과 | random_state 고정 필요 |
| 군집 | 흐릿 | 뚜렷 |
| 용도 | 전체 분포 빠르게 | 예쁜 시각화 |

**주의**: 데이터 포인트가 적으면(여기선 16개) 토픽별로 옹기종기 모이지 않을 수 있음.
t-SNE는 재실행마다 약간씩 달라지므로, 팀장님께 보여드리고 또 보여드릴 때 그림이 달라지면 곤란하니 **`random_state` 고정**!

## 6. FAISS - Facebook AI Similarity Search

### 왜 FAISS?

지금까지는 벡터 수가 적으니 `cosine_similarity`를 직접 계산했지만,
실전에선 **수백만 벡터**에서 쿼리와 가까운 top-k를 찾아야 함.

- FAISS = Meta(구 Facebook)가 만든 **고속 벡터 유사도 검색 라이브러리**
- 인덱싱 + 근사 최근접 이웃(ANN) 기술로 수백만 벡터에서 밀리초 단위 검색
- LangChain의 `FAISS` vectorstore가 내부적으로 이 라이브러리 사용

> **비유**: 도서관 책 100만 권 중 내 관심사와 비슷한 걸 찾아주는 **초고속 사서**. 카테고리/인덱스를 미리 만들어둬서 바로 찾음.

In [ ]:
# FAISS 설치 (Colab에서는 faiss-cpu)
!pip install -q langchain_community faiss-cpu

In [ ]:
# FAISS vectorstore 임포트
from langchain_community.vectorstores import FAISS

# FAQ 문서 10개 - 주택청약이 아닌 LLM 관련 일반 FAQ
faq_docs = [
    "LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다.",
    "RAG는 외부 문서를 검색하여 LLM 답변 정확도를 높이는 기술입니다.",
    "벡터 데이터베이스는 임베딩 벡터를 효율적으로 저장하고 검색합니다.",
    "프롬프트 엔지니어링은 AI에게 효과적인 지시를 작성하는 기술입니다.",
    "파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습시킵니다.",
    "토큰은 LLM이 텍스트를 처리하는 기본 단위입니다.",
    "임베딩은 텍스트를 수치 벡터로 변환하는 기술입니다.",
    "LCEL은 LangChain의 체인 구성을 위한 표현식 언어입니다.",
    "에이전트는 LLM이 도구를 사용해서 자율적으로 작업하는 시스템입니다.",
    "Gradio는 ML 모델을 위한 간단한 웹 인터페이스를 만드는 라이브러리입니다.",
]

In [ ]:
# FAISS.from_texts: 텍스트 리스트 + 임베딩 모델 -> 벡터스토어 자동 생성
# 내부에서 각 텍스트를 임베딩 -> FAISS 인덱스에 저장
vectorstore = FAISS.from_texts(faq_docs, embeddings_model)

# 여러 쿼리로 검색해보기
queries = [
    'RAG가 뭐예요?',
    'AI 챗봇을 어떻게 만드나요?',
    '토큰 비용을 줄이고 싶어요'
]

for query in queries:
    # similarity_search_with_score: 유사 문서 top-k + 거리 점수 리턴
    # 주의: FAISS 기본은 L2 거리(낮을수록 유사), cosine 아님
    results = vectorstore.similarity_search_with_score(query, k=3)
    for doc, score in results:
        print(f' [{score}] {doc.page_content}')
    print(" ===== ")

### FAISS 점수 해석 주의

- `similarity_search_with_score`의 score는 **L2(유클리드) 거리**가 기본
- **낮을수록 유사** (0에 가까울수록 좋음)
- cosine과 헷갈리지 말 것! 코사인으로 쓰려면 `distance_strategy` 옵션 조정

## 7. Document + Metadata

텍스트만 저장하면 나중에 "출처가 뭐지?", "언제 작성된 문서지?" 못 알아봄.
**Document** 객체로 본문 + 메타데이터를 묶어 저장하면 필터링, 출처 추적 가능.

> **비유**: 책에 "제목/저자/출판년도" 라벨을 붙이는 것. 내용만 복사해서 스크랩하면 나중에 어디서 왔는지 모름.

In [ ]:
# LangChain의 Document 객체: page_content + metadata(dict)
from langchain_core.documents import Document

In [ ]:
# 각 문서에 category, source, year 메타데이터 부여
docs_with_meta = [
    Document(page_content="LangChain은 LLM 앱 개발 프레임워크입니다.", metadata={"category": "tech", "source": "doc1", "year": 2024}),
    Document(page_content="RAG는 외부 문서를 검색해 답변 품질을 높이는 기술입니다.", metadata={"category": "tech", "source": "doc2", "year": 2024}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 검색합니다.", metadata={"category": "tech", "source": "doc3", "year": 2023}),
    Document(page_content="GPT-4는 OpenAI가 개발한 대규모 언어 모델입니다.", metadata={"category": "model", "source": "doc4", "year": 2023}),
    Document(page_content="프롬프트 엔지니어링은 AI에게 효과적인 지시를 작성하는 기술입니다.", metadata={"category": "technique", "source": "doc5", "year": 2024}),
    Document(page_content="파인튜닝은 사전학습 모델을 특정 데이터에 맞게 재학습시킵니다.", metadata={"category": "technique", "source": "doc6", "year": 2023}),
    Document(page_content="LangGraph는 LLM 워크플로우를 그래프 구조로 설계하는 라이브러리입니다.", metadata={"category": "tech", "source": "doc7", "year": 2024}),
    Document(page_content="임베딩은 텍스트를 숫자 벡터로 변환하는 과정입니다.", metadata={"category": "concept", "source": "doc8", "year": 2023}),
    Document(page_content="Chroma는 로컬에서 사용 가능한 오픈소스 벡터 데이터베이스입니다.", metadata={"category": "tool", "source": "doc9", "year": 2024}),
    Document(page_content="Agent는 LLM이 스스로 도구를 선택하고 실행하는 구조입니다.", metadata={"category": "concept", "source": "doc10", "year": 2024}),
]

# FAISS.from_documents: Document 리스트를 그대로 받음 (metadata 유지)
vs_meta = FAISS.from_documents(docs_with_meta, embeddings_model)

In [ ]:
# similarity_search (score 없는 버전)로 상위 3개 검색
results = vs_meta.similarity_search('AI 개발 도구', k=3)
results

In [ ]:
# 메타데이터를 함께 출력 - 출처/년도 추적 가능!
for doc in results:
    print(f'[{doc.metadata["category"]}/{doc.metadata["source"]}/{doc.metadata["year"]}], [{doc.page_content}]')

## 8. 다국어 임베딩 - 한↔영 교차 검색

### 멀티링구얼(multilingual) 임베딩이란?

- `text-embedding-3-small`은 **다국어 지원** 임베딩 모델
- "나는 학교에 간다" 와 "I go to school" 이 **같은 벡터 공간**의 **가까운 점**에 찍힘
- → 한국어로 검색해도 영어 문서에서 관련 내용을 찾을 수 있음!

> **비유**: 국제공항 표지판이 "한/영/일/중" 병기되어 있는 것처럼, 벡터 공간에 모든 언어의 "출구" 표시가 같은 위치에 있는 셈.

### 모든 임베딩 모델이 다국어인 건 아님
- 모델이 학습 때 여러 언어를 봐야 다국어 가능
- 영어 전용 모델에 한국어를 넣으면 이상한 결과 나옴

In [ ]:
# Document 객체로 4개 언어(한/영/일/불) 두 주제(LangChain, Vector DB) 교차 구성
multilingual = [
    Document(page_content="LangChain은 LLM 앱 개발 프레임워크입니다.",      metadata={"lang": "ko"}),
    Document(page_content="LangChain is a framework for LLM applications.", metadata={"lang": "en"}),
    Document(page_content="LangChainはLLMアプリ開発フレームワークです。",     metadata={"lang": "ja"}),
    Document(page_content="LangChain est un framework pour les LLM.",        metadata={"lang": "fr"}),
    Document(page_content="벡터 데이터베이스는 임베딩을 저장합니다.",            metadata={"lang": "ko"}),
    Document(page_content="Vector databases store embeddings.",              metadata={"lang": "en"}),
    Document(page_content="ベクターデータベースは埋め込みを保存します。",         metadata={"lang": "ja"}),
    Document(page_content="Les bases vectorielles stockent les embeddings.", metadata={"lang": "fr"}),
]

# Document 리스트에서 본문/라벨 분리
texts = [doc.page_content for doc in multilingual]
langs = [doc.metadata["lang"] for doc in multilingual]

# 8개 문장 임베딩 -> 8x8 유사도 행렬
embs_multi = embeddings_model.embed_documents(texts)
sim_matrix = cosine_similarity(embs_multi)
sim_matrix

In [ ]:
# 히트맵 + PCA 시각화 - 주제별 색깔 다르게
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. 히트맵 - 8x8 유사도 행렬
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# 라벨 (언어 + 문장 앞 10글자)
labels = [f"[{doc.metadata['lang']}] {doc.page_content[:10]}" for doc in multilingual]

# 히트맵: 빨강(낮음) -> 초록(높음)
# -> 같은 주제(예: 모든 'LangChain' 문장들)끼리 언어 달라도 유사도 높음!
sns.heatmap(
    sim_matrix,
    annot=True, fmt=".2f",
    cmap="RdYlGn",
    xticklabels=labels, yticklabels=labels,
    vmin=-1, vmax=1,
    ax=axes[0]
)
axes[0].set_title("문서 간 코사인 유사도 히트맵", fontsize=14)
axes[0].tick_params(axis='x', rotation=45)

# 2. PCA 2차원 시각화 - 언어별로 색깔 다르게
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(np.array(embs_multi))

lang_colors = {"ko": "#FF6B6B", "en": "#4ECDC4", "ja": "#FFD93D", "fr": "#6C5CE7"}

for i, doc in enumerate(multilingual):
    lang = doc.metadata["lang"]
    axes[1].scatter(coords_2d[i, 0], coords_2d[i, 1],
                    c=lang_colors[lang], s=150, label=lang)
    axes[1].annotate(
        f"[{lang}] {doc.page_content[:10]}",
        (coords_2d[i, 0], coords_2d[i, 1]),
        fontsize=9, ha='right', va='bottom'
    )

# 범례 중복 제거 (같은 lang 여러 번 나오므로)
handles, labels_legend = axes[1].get_legend_handles_labels()
unique = dict(zip(labels_legend, handles))
axes[1].legend(unique.values(), unique.keys(), fontsize=12)
axes[1].set_title("PCA 2D 시각화 - 언어별 분포", fontsize=14)

plt.tight_layout()
plt.show()

### 히트맵 해석 팁

- **같은 주제, 다른 언어** (예: LangChain ko vs en vs ja vs fr) → 유사도 ↑ (초록)
- **다른 주제, 같은 언어** (예: ko LangChain vs ko 벡터DB) → 유사도 ↓ (빨강/노랑)
- → 임베딩은 **언어가 아닌 '의미'** 를 기준으로 벡터 공간에 배치!

## 9. 교차 언어 검색 - 한국어 쿼리 → 영어 문서

실전 시나리오: 영어로 된 공식 문서가 있는데, 한국인 사용자가 한국어로 질문.

In [ ]:
# 영어 문서들로 벡터 스토어 구성
english_docs = [
    "LangChain is a framework for building LLM applications.",
    "RAG improves LLM accuracy by retrieving external documents.",
    "Vector databases store and search embedding vectors efficiently.",
    "Fine-tuning adapts a pre-trained model to specific data.",
    "Prompt engineering is the art of crafting effective instructions for AI.",
]

# FAISS 스토어 생성
vs_en = FAISS.from_texts(english_docs, embeddings_model)

# 한국어 질문으로 영어 문서 검색!
korean_queries = [
    "RAG가 뭐에요?", "벡터 데이터베이스 설명해주세요", "프롬프트 엔지니어링은?"
]

for q in korean_queries:
    results = vs_en.similarity_search_with_score(q, k=3)
    print(f"\nKR {q}")
    for doc, score in results:
        print(f" US {score}, {doc.page_content}")

### 관찰 포인트

- 한국어 쿼리 "RAG가 뭐에요?" → 영어 문서 "RAG improves LLM accuracy..." 가 1위로 나옴!
- 다국어 임베딩 덕분에 **번역 없이 교차 언어 검색** 가능
- 실전에선 다국어 사용자 지원, 글로벌 문서 베이스 검색 등에 유용

## 정리

오늘 익힌 것:

1. **CacheBackedEmbeddings** 실측 - 2차 호출은 수백 배 빠름
2. **배치 처리** (`process_documents`) - API 호출 수 줄여 속도/비용 개선
3. **Cosine Similarity 엣지 케이스** - 반의어에도 유사도가 높게 나오는 이유는 "주제 공유"
4. **find_most_similar v1 vs v2** - 쿼리+코퍼스 일괄 임베딩이 훨씬 효율
5. **PCA vs t-SNE** - PCA는 빠르고 재현성, t-SNE는 군집이 뚜렷
6. **FAISS vectorstore** - 대규모 벡터 검색용 라이브러리
7. **Document + metadata** - 본문+출처/카테고리 함께 저장
8. **다국어 임베딩** - 같은 의미면 언어 달라도 벡터 공간에서 가까움
9. **교차 언어 검색** - 한국어 쿼리로 영어 문서 검색 성공

### 실전에서 주의할 점

- LangChain 라이브러리 문법이 **버전마다 자주 바뀜** (import 경로 달라지거나)
- 해결책: `conda` 환경 + `requirements.txt` 고정, 테스트 스크립트로 회귀 검증
- FAISS score는 **L2 거리** 기본이라 "낮을수록 유사" (cosine과 반대 해석)
- `InMemoryByteStore`는 휘발성 → 영구 저장 원하면 다른 ByteStore 사용

### 다음 시간 (3/19~20)
- Document Loader로 실제 파일(PDF/CSV/HTML) 로딩
- 다양한 TextSplitter 전략
- FAISS 인덱스 저장/로드 (`save_local` / `load_local`)
- LCEL로 RAG 체인 구성: `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()`